# YOLO26n `best.pt` → ESP-DL `.espdl` para ESP32-S3

Cuaderno único para Google Colab (Python 3.12). **No entrena el modelo**: recibe los pesos `best.pt` de un YOLO26n de Ultralytics y un conjunto de imágenes representativas para calibración.

La ruta por defecto (`RUN_MODE = "full"`) sigue el tutorial oficial de Espressif para YOLO26n: exportación ONNX con el parche de la cabeza, PTQ, TQT, fusión LUT INT16, cirugía del grafo y exportación `.espdl` para `esp32s3`.

`RUN_MODE = "autoquant"` es una rama experimental que usa `espdl_auto_quantize_onnx` para buscar configuraciones; sin `evaluate_fn` ordena por error de cuantización, no por mAP, y no sustituye la fusión LUT específica de la ruta oficial.

## Decisiones importantes

- `IMG_SIZE = 512` es la resolución de trabajo del modelo exportado. La resolución del firmware actual sigue en `224`; el puente ESP se adaptará posteriormente.
- `512x512` aumenta el consumo de memoria; el cuaderno usa `BATCH_SIZE = 1` para reducir el pico de VRAM. La partición y el puente del ESP32-S3 se ajustarán en una etapa posterior.
- `best.pt` debe ser realmente YOLO26n (`reg_max = 1`). El cuaderno aborta si la arquitectura no coincide.
- La exportación fuerza `batch=1` y shapes estáticos porque ESP-DL no admite batch dinámico/multiple.
- Las imágenes de calibración no necesitan etiquetas. Deben ser imágenes RGB representativas del despliegue.
- Se usa CUDA automáticamente para PyTorch/ESP-PPQ cuando Colab tiene GPU (`DEVICE`). `TorchExecutor` además usa `device='cuda'` por defecto en esp-ppq, así que PTQ y TQT corren en GPU sin configuración extra.
- `TQT_COLLECTING_DEVICE` sigue a la GPU si existe. El tutorial oficial usa `'cpu'` porque `TrainingBasedPass.collect()` baja todos los pesos a CPU en cada bloque y los vuelve a subir en cada Conv, lo que hace el TQT mucho más lento. Con `'cuda'` desaparece ese ida y vuelta; a cambio la caché de activaciones de TQT vive en la VRAM. Si aparece `OutOfMemoryError`, baje `MAX_CALIB_IMAGES` o vuelva a `'cpu'`.
- `BATCH_SIZE = 4` con `CALIB_STEPS = 32` mantiene las mismas 128 imágenes y los mismos 32 pasos de calibración, pero reduce a la cuarta parte los forwards de TQT, que es la etapa dominante.
- El objetivo del modelo es **ESP32-S3**; no se debe reutilizar un `.espdl` generado para ESP32-P4.
- La ruta híbrida INT8/INT16 + LUT es la validada por el tutorial actual y requiere ESP-DL `>=3.3.11`. Si debe conservarse un runtime `3.3.10`, ponga `USE_MIXED_INT16=False` y `USE_TQT_LUT=False` para obtener una prueba INT8 más simple (aun así requiere adaptar el contrato de salida).
- Se instala `esp-ppq==1.3.11`. El `requirements.txt` del tutorial oficial fija `esp-ppq==1.2.4`; se verificó que `onnx_parser.py`, `prepare_input()` y `dummy_forward()` son idénticos en 1.2.4 y 1.3.11, así que la ruta `RUN_MODE='full'` no depende de la versión. 1.3.11 se mantiene solo porque aporta `espdl_auto_quantize_onnx` para la rama `autoquant`.
- Las imágenes de calibración se fuerzan a RGB. `CaliDataset` del tutorial solo convertía el modo `"L"`, de modo que los PNG con canal alfa (`"RGBA"`, 4 canales) y los de paleta (`"P"`, 1 canal) llegaban al grafo y la primera Conv abortaba con `expected input[1, 1, H, W] to have 3 channels`. Es el defecto reportado en https://github.com/espressif/esp-dl/issues/329, ya reconocido por Espressif.
- NumPy y sus paquetes binarios se fijan a versiones compatibles con Python 3.12 (`numpy==1.26.4`, `scipy==1.14.1`, `opencv-python-headless==4.11.0.86`, `matplotlib==3.9.4`) para evitar el conflicto ABI de NumPy 1.x/2.x en Colab.
- La celda 3 aplica un fix de compatibilidad para `esp-ppq 1.3.x`: conserva el nombre canónico devuelto por la deduplicación de LUT antes de la exportación `.espdl`.

## Entradas

1. Coloque `best.pt` en `/content/best.pt` (o edite `WEIGHTS_PATH`).
2. Para Roboflow, edite `ENV_PATH` para apuntar al `.env` de Google Drive y complete `ROBOFLOW_WORKSPACE`, `ROBOFLOW_PROJECT` y `ROBOFLOW_VERSION` si no están en ese archivo. Variables aceptadas: `ROBOFLOW_API_KEY`, `ROBOFLOW_WORKSPACE`, `ROBOFLOW_PROJECT` y `ROBOFLOW_VERSION`. El cuaderno monta Drive, descarga el dataset y no imprime la API key.
3. Si `USE_ROBOFLOW=False`, coloque imágenes de calibración en una carpeta, por ejemplo `/content/dataset/images/train`, o edite `CALIB_IMAGE_DIR`.
4. Ejecute las celdas en orden. La celda de dependencias instala `onnxscript` y fuerza `dynamo=False` para evitar el exportador ONNX nuevo de PyTorch 2.9+. También limpia una posible mezcla ABI de NumPy, instala una copia aislada de `numpy==1.26.4` y valida `numpy.random`; si aparece `numpy.dtype size changed`, use **Runtime → Reiniciar sesión** y ejecute nuevamente desde la celda 1. Al finalizar se descarga un ZIP con `.espdl`, `.info`, `.json` y el reporte.

## Ejecución

Ejecute **Runtime → Reiniciar sesión** y luego **Run all** en orden, sin reejecutar celdas intermedias. `load_onnx_graph()` no tiene caché: si se vuelve a llamar, reemplaza el grafo cuantizado por uno nuevo y sin cuantizar, y el resultado final mide un modelo que en realidad es FP32 (ver https://github.com/espressif/esp-dl/issues/325). `executor.tracing_operation_meta()` debe ejecutarse antes de cualquier pasada que dependa de formas, porque el parser ONNX de ESP-PPQ no lee `value_info` y deja `shape=None` en todas las variables.

## Problemas upstream conocidos revisados

| Issue | Qué cubre | Estado en este cuaderno |
|---|---|---|
| [#329](https://github.com/espressif/esp-dl/issues/329) | `CaliDataset` solo convierte imágenes `"L"`; `"RGBA"`/`"P"` rompen la calibración | Corregido en la celda 4 (conversión incondicional a RGB + validación) |
| [#325](https://github.com/espressif/esp-dl/issues/325) | Pérdida de `tracing_operation_meta` y celdas de recarga duplicadas | Celda 7 blinda el tracing y verifica la forma resultante |
| [#288](https://github.com/espressif/esp-dl/issues/288) | El ONNX de YOLO26 de Ultralytics incluye el postproceso de bbox y rompe `/model.23/Concat` | Celda 5 exporta con `ESP_YOLO` (`scripts/export.py`), que quita el postproceso |
| [#328](https://github.com/espressif/esp-dl/issues/328) | `AddLUTPattern` ignora el nombre canónico que devuelve `add_lut()` | Celda 3 ya instala `_espdl_add_lut_export_fixed` |
| [#322](https://github.com/espressif/esp-dl/issues/322) | `ImportError` de `TrainedQuantizationThresholdPass` por import circular | No aplica: la celda 7 importa desde el paquete `esp_ppq.quantization.optim` |

## Advertencia de integración con el proyecto actual

Este cuaderno **no modifica el firmware**. El modelo oficial YOLO26 de ESP-DL expone seis salidas con nombres `one2one_p{3,4,5}_{box,cls}` y debe consumirse con el procesador YOLO26 de ESP-DL (preprocesamiento con letterbox). El `.espdl` de este cuaderno se exporta ahora para `512x512`; el firmware actual continúa usando `224`, una entrada INT8 y una salida `output0`, por lo que el modelo nuevo no debe sustituir al actual hasta adaptar ese puente. No se incluye un modo `output0` porque no es el contrato del tutorial oficial y requiere una conversión y un firmware distintos. Además, la partición `app0` actual es de 5 MiB; el reporte comprueba el tamaño y avisa si el `.espdl` no cabe. El reporte final incluye esta advertencia.

## Referencias oficiales

- Tutorial YOLO26n: https://github.com/espressif/esp-dl/tree/4f4efd7ff021a2c9dcafad4f1dab3da349b8bf4a/examples/tutorial/how_to_quantize_model/quantize_yolo26
- Cuantización ESP-DL: https://docs.espressif.com/projects/esp-dl/en/latest/tutorials/how_to_quantize_model.html
- AutoQuant: https://docs.espressif.com/projects/esp-dl/en/latest/tutorials/auto_quantization/how_to_use_AutoQuant.html
- Procesador YOLO26: https://github.com/espressif/esp-dl/tree/4f4efd7ff021a2c9dcafad4f1dab3da349b8bf4a/models/yolo26
- Componente ESP-DL (registry; verificar versión): https://components.espressif.com/components/espressif/esp-dl


In [ ]:
# @title 1. Dependencias oficiales de Espressif
import importlib.util
import subprocess
import sys
import shutil
from pathlib import Path


def pip_install(*packages):
    print('Instalando/verificando:', ' '.join(packages), flush=True)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])


if importlib.util.find_spec('torch') is None:
    pip_install('torch>=2.4.0', 'torchvision>=0.19.0')

# No dejamos dos instalaciones de onnxruntime compitiendo.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'onnxruntime'], check=False)
# Limpia una posible mezcla de ABI NumPy 1.x/2.x tras una actualización parcial.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy', 'numpy-base'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'opencv-python', 'opencv-python-headless'], check=False)
# Se instala una copia aislada de NumPy para no depender de archivos residuales del runtime.
NUMPY_TARGET = Path('/content/esp_numpy126')
if NUMPY_TARGET.exists():
    shutil.rmtree(NUMPY_TARGET)
pip_install('--no-cache-dir', '--no-deps', '--target', str(NUMPY_TARGET), 'numpy==1.26.4')
sys.path.insert(0, str(NUMPY_TARGET))
for module_name in list(sys.modules):
    if module_name == 'numpy' or module_name.startswith('numpy.'):
        del sys.modules[module_name]
onnxsim_package = 'onnxsim-prebuilt==0.4.39.post2' if sys.version_info >= (3, 12) else 'onnxsim==0.4.36'
pip_install(
    'numpy==1.26.4',
    # El requirements.txt del tutorial oficial fija 1.2.4 y esa version ya trae
    # TQT. 1.3.11 (== master, tag del 2026-09-02) se usa solo porque aporta
    # espdl_auto_quantize_onnx para RUN_MODE='autoquant'. Verificados identicos
    # en 1.2.4 y 1.3.11: onnx_parser.py, prepare_input() y dummy_forward().
    'esp-ppq==1.3.11',
    'onnx==1.17.0',
    'scipy==1.14.1',
    onnxsim_package,
    'onnxscript==0.7.2',
    'ultralytics==8.4.7',
    'opencv-python-headless==4.11.0.86',
    'Pillow>=9.4',
    'PyYAML>=6.0',
    'matplotlib==3.9.4',
    'tqdm>=4.66',
    'ninja>=1.11',
    'roboflow>=1.2.13',
    'python-dotenv>=1.0',
)

import numpy as np
from numpy.random import RandomState  # valida la ABI de NumPy antes de cargar los helpers
print('NumPy:', np.__version__, '| ABI OK')
print('NumPy path:', np.__file__)
import torch
if torch.cuda.is_available():
    # esp-ppq puede haber instalado la distribución CPU como dependencia;
    # se reemplaza por la variante GPU para la validación ONNX.
    subprocess.run(
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'],
        check=False,
    )
    try:
        pip_install('onnxruntime-gpu>=1.19.0')
    except Exception:
        pip_install('onnxruntime>=1.19.0')
else:
    pip_install('onnxruntime>=1.19.0')

import importlib
importlib.invalidate_caches()
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# @title 2. Configuración y localización de best.pt / imágenes
from pathlib import Path
import os
import random
import re
import sys
import json
import math
import shutil
import zipfile
import hashlib
import types

ROOT = Path('/content/drive/MyDrive/Colab Notebooks')
OUT_DIR = ROOT / 'output'
ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Edite estas dos rutas si sus archivos están en otra ubicación.
WEIGHTS_PATH = Path('ruta_donde_los_pesos.pt')
CALIB_IMAGE_DIR = Path('ruta_data_set_calibracion')
DATA_YAML = Path('/content/(nombre_del_proyecto)/data.yaml')

# Dataset Roboflow. La API key se lee desde Drive y nunca se imprime.
USE_ROBOFLOW = True
ENV_PATH = Path('ruta_del_archivo_.env')  # edite si el archivo está en otra carpeta
ROBOFLOW_DOWNLOAD_DIR = ROOT / 'roboflow_dataset'
ROBOFLOW_FORMAT = 'yolo26'                      # use 'yolov8' si esa es la exportación de su proyecto
ROBOFLOW_WORKSPACE = 'nombre_del_workspace'                         # opcional si está en .env
ROBOFLOW_PROJECT = 'pryecto_ID'                            # slug o project ID; opcional si está en .env
ROBOFLOW_VERSION = '1'                            # por ejemplo 3; opcional si está en .env

# Ponga True para que se abra el selector de archivos de Colab.
UPLOAD_INPUTS = False
RUN_MODE = 'full'                 # 'full' recomendado; 'autoquant' como alternativa
IMG_SIZE = 320                    # <--- La resolucion
MAX_CALIB_IMAGES = 128
BATCH_SIZE = 4                     # 4 = mismo dataset, ~4x menos forwards en TQT
CALIB_STEPS = 32                   # 32 batches x 4 = 128 imágenes de calibración
USE_MIXED_INT16 = True           # False para una línea base INT8 sin w8a16/LUT
USE_TQT_LUT = True
TQT_STEPS = 100                   # súbelo a 200 para una búsqueda más larga
TQT_BLOCK_SIZE = 4                # 4 = valor del tutorial oficial; 1 = un bloque por op (muy lento)
# esp-ppq compara este valor contra las cadenas 'cpu' y 'cuda', asi que aqui NO
# se usa un indice entero: el '0' del device=0 de Ultralytics es otra cosa.
# Con 'cuda', TQT mantiene pesos y cache de activaciones en la GPU en vez de
# bajarlos a CPU en cada bloque. Si agota la VRAM, bajarlo a 'cpu' o reducir
# MAX_CALIB_IMAGES.
TQT_COLLECTING_DEVICE = 'cuda' if HAS_GPU else 'cpu'
INT16_LUT_STEP = 32
TARGET = 'esp32s3'
SEED = 1234
AUTOQUANT_USE_GPU = True
AUTOQUANT_CANDIDATES = 3
DOWNLOAD_RESULT = True
CURRENT_APP_PARTITION_BYTES = 0x500000  # 5 MiB en el proyecto actual; solo informativo

if UPLOAD_INPUTS:
    from google.colab import files
    uploaded = files.upload()
    for name, payload in uploaded.items():
        destination = Path('/content') / name
        destination.write_bytes(payload)
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(destination) as archive:
                archive.extractall('/content/dataset_unpacked')
    WEIGHTS_PATH = Path('/content') / next(
        (n for n in uploaded if n.lower().endswith(('.pt', '.pth'))), WEIGHTS_PATH.name
    )
if USE_ROBOFLOW:
    from google.colab import drive

    # Intentar limpiar el punto de montaje si hay un error persistente
    try:
        drive.flush_and_unmount()
    except Exception:
        pass

    mount_point = '/content/drive'
    # Si la carpeta de montaje contiene archivos residuales y no se puede montar, la limpiamos de raíz
    if os.path.exists(mount_point) and os.path.isdir(mount_point):
        if os.listdir(mount_point):
            print(f"Limpiando archivos conflictivos en {mount_point}...")
            try:
                shutil.rmtree(mount_point)
            except Exception as e:
                print(f"No se pudo limpiar la carpeta {mount_point}: {e}. Intentando montar de todos modos.")

    try:
        drive.mount(mount_point, force_remount=True)
    except Exception as e:
        print(f"Error al montar en {mount_point}: {e}. Probando punto de montaje alternativo...")
        mount_point = '/content/gdrive'
        drive.mount(mount_point, force_remount=True)
        # Ajustamos las rutas globales para que apunten al nuevo punto de montaje si cambió
        ROOT = Path(str(ROOT).replace('/content/drive', '/content/gdrive'))
        OUT_DIR = Path(str(OUT_DIR).replace('/content/drive', '/content/gdrive'))
        WEIGHTS_PATH = Path(str(WEIGHTS_PATH).replace('/content/drive', '/content/gdrive'))
        CALIB_IMAGE_DIR = Path(str(CALIB_IMAGE_DIR).replace('/content/drive', '/content/gdrive'))
        ENV_PATH = Path(str(ENV_PATH).replace('/content/drive', '/content/gdrive'))
        ROBOFLOW_DOWNLOAD_DIR = Path(str(ROBOFLOW_DOWNLOAD_DIR).replace('/content/drive', '/content/gdrive'))

    from dotenv import dotenv_values

    if not ENV_PATH.exists():
        raise FileNotFoundError(
            f'No se encontró el archivo .env en {ENV_PATH}. Edite ENV_PATH en la celda 2.'
        )
    env_values = dotenv_values(ENV_PATH)

    def env_value(*names):
        for name in names:
            value = os.getenv(name) or env_values.get(name)
            if value:
                return value
        return None

    api_key = env_value('ROBOFLOW_API_KEY', 'ROBOFLOW_KEY')
    workspace = ROBOFLOW_WORKSPACE or env_value('ROBOFLOW_WORKSPACE', 'WORKSPACE_ID')
    project = ROBOFLOW_PROJECT or env_value('ROBOFLOW_PROJECT', 'ROBOFLOW_PROJECT_ID', 'PROJECT_ID')
    version = ROBOFLOW_VERSION or env_value('ROBOFLOW_VERSION', 'DATASET_VERSION', 'VERSION')
    missing = [
        name for name, value in {
            'ROBOFLOW_API_KEY': api_key,
            'ROBOFLOW_WORKSPACE': workspace,
            'ROBOFLOW_PROJECT': project,
            'ROBOFLOW_VERSION': version,
        }.items() if not value
    ]
    if missing:
        raise ValueError(
            'Faltan datos de Roboflow: ' + ', '.join(missing) +
            '. Edite las variables de la celda 2 o el archivo .env.'
        )

    from roboflow import Roboflow
    rf = Roboflow(api_key=api_key)
    roboflow_project = rf.workspace(workspace).project(project)
    roboflow_version = roboflow_project.version(int(version))
    dataset = roboflow_version.download(
        ROBOFLOW_FORMAT,
        location=str(ROBOFLOW_DOWNLOAD_DIR),
    )
    downloaded_root = Path(getattr(dataset, 'location', ROBOFLOW_DOWNLOAD_DIR))
    downloaded_yaml = downloaded_root / 'data.yaml'
    if downloaded_yaml.exists():
        DATA_YAML = downloaded_yaml
    for candidate in [
        downloaded_root / 'train' / 'images',
        downloaded_root / 'train',
        downloaded_root / 'images' / 'train',
        downloaded_root,
    ]:
        if candidate.exists():
            CALIB_IMAGE_DIR = candidate
            break
    print('Dataset Roboflow descargado en:', downloaded_root)

if not WEIGHTS_PATH.exists():
    candidates = sorted(Path('/content').glob('*.pt')) + sorted(Path('/content').glob('*.pth'))
    if candidates:
        WEIGHTS_PATH = candidates[0]
if not WEIGHTS_PATH.exists():
    raise FileNotFoundError('No se encontró best.pt. Edite WEIGHTS_PATH o suba el archivo.')

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

def find_images(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(
        p for p in root.rglob('*')
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

if not CALIB_IMAGE_DIR.exists():
    for candidate in [ROBOFLOW_DOWNLOAD_DIR, Path('/content/dataset'), Path('/content/dataset_unpacked'), Path('/content')]:
        found = find_images(candidate)
        if found:
            CALIB_IMAGE_DIR = candidate
            break

images = find_images(CALIB_IMAGE_DIR)
if not images:
    raise FileNotFoundError(
        'No se encontraron imágenes. Revise ENV_PATH y los datos Roboflow, '
        'o edite CALIB_IMAGE_DIR / ponga USE_ROBOFLOW=False y use una carpeta local.'
    )

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
HAS_GPU = torch.cuda.is_available()
if RUN_MODE.lower() not in {'full', 'autoquant'}:
    raise ValueError("RUN_MODE debe ser 'full' o 'autoquant'.")
if TARGET != 'esp32s3':
    raise ValueError('Este cuaderno está configurado para TARGET = esp32s3.')
if IMG_SIZE % 32 != 0:
    raise ValueError('IMG_SIZE debe ser múltiplo de 32 (por ejemplo 512, 320 o 640).')

print('best.pt:', WEIGHTS_PATH)
print('Imágenes de calibración:', len(images), 'en', CALIB_IMAGE_DIR)
print('data.yaml opcional:', DATA_YAML if DATA_YAML.exists() else 'no encontrado')
print('Dispositivo:', DEVICE, '| TQT:', QATConfig.TQT_COLLECTING_DEVICE if 'QATConfig' in globals() else TQT_COLLECTING_DEVICE, '| imagen:', IMG_SIZE, '| modo:', RUN_MODE)


In [ ]:
# @title 3. Descargar el código auxiliar oficial (commit fijo)
import urllib.request

ESP_DL_REF = '4f4efd7ff021a2c9dcafad4f1dab3da349b8bf4a'
ESP_DL_RAW = f'https://raw.githubusercontent.com/espressif/esp-dl/{ESP_DL_REF}/examples/tutorial/how_to_quantize_model/quantize_yolo26'
HELPERS = Path('/content/esp_dl_yolo26_helpers')
HELPERS.mkdir(parents=True, exist_ok=True)

helper_files = [
    'scripts/__init__.py',
    'scripts/export.py',
    'scripts/notebook_helpers.py',
    'scripts/utils.py',
    'scripts/dataset.py',
    'scripts/esp_ppq_patch.py',
    'scripts/esp_ppq_patch_2.py',
    'esp_ppq_lut/__init__.py',
    'esp_ppq_lut/emulator.py',
    'esp_ppq_lut/exporter.py',
    'esp_ppq_lut/passes.py',
    'esp_ppq_lut/patches.py',
    'esp_ppq_lut/utils.py',
    'esp_ppq_lut/verifier.py',
]

for relative in helper_files:
    destination = HELPERS / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        urllib.request.urlretrieve(f'{ESP_DL_RAW}/{relative}', str(destination))
        print('Descargado:', relative)

sys.path.insert(0, str(HELPERS))
sys.path.insert(0, str(HELPERS / 'scripts'))

from esp_ppq.api import get_target_platform, AutoQuantSearchSetting, espdl_auto_quantize_onnx
from esp_ppq.quantization.optim import TrainedQuantizationThresholdPass

# Ultralytics 8.4.7 puede pedir la distribución ``onnxsim`` aunque en Python
# 3.12 el wheel funcional es ``onnxsim-prebuilt``. El módulo importable sigue
# siendo ``onnxsim``; desactivamos los chequeos automáticos para que no reinstale
# onnxruntime-gpu durante la exportación. La cuantización GPU usa PyTorch/ESP-PPQ.
import importlib
import os
os.environ['ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS'] = '1'
import ultralytics.utils.checks as _ultralytics_checks

# No dejamos que Ultralytics autoinstale requisitos durante la exportación.
# Si una ejecución anterior dejó el wrapper antiguo, recargamos el módulo
# para eliminarlo antes de continuar.
if getattr(_ultralytics_checks.check_requirements, '__name__', '') == '_check_requirements_python312':
    importlib.reload(_ultralytics_checks)
print('ULTRALYTICS_SKIP_REQUIREMENTS_CHECKS=1')
import onnxsim

class QATConfig:
    IMG_SZ = IMG_SIZE
    DEVICE = DEVICE
    DATA_YAML_FILE = str(DATA_YAML) if DATA_YAML.exists() else ''
    DATA_FRACTION = 1.0
    SEED = SEED
    BATCH_SIZE = BATCH_SIZE
    CALIB_MAX_IMAGES = MAX_CALIB_IMAGES
    CALIB_VALID_EXTENSIONS = tuple(sorted(IMAGE_EXTENSIONS))
    DATA_FALLBACK_PATH = str(CALIB_IMAGE_DIR)
    CALIB_STEPS = CALIB_STEPS
    QUANT_CALIB_METHOD = 'percentile'
    QUANT_ALIGNMENT = 'Align to Output'
    TARGET_PLATFORM = get_target_platform(TARGET, 8)
    INT16_LUT_STEP = INT16_LUT_STEP
    TQT_STEPS = TQT_STEPS
    TQT_LR = 2e-5
    TQT_INT_LAMBDA = 0.25
    TQT_BLOCK_SIZE = TQT_BLOCK_SIZE
    TQT_COLLECTING_DEVICE = TQT_COLLECTING_DEVICE
    BASE_DIR = str(ROOT)
    MODEL_NAME = WEIGHTS_PATH.stem
    PT_FILE = str(WEIGHTS_PATH)
    ONNX_FILE = 'yolo26n_export.onnx'
    ONNX_PATH = str(ROOT / ONNX_FILE)
    ESPDL_OUTPUT_DIR = str(OUT_DIR)
    LOSS_DEFAULTS = {}
    VAL_PLOT_MAX_BATCHES = 0

config_module = types.ModuleType('config')
config_module.QATConfig = QATConfig
sys.modules['config'] = config_module

from utils import seed_everything, register_mod_op, get_exclusive_ancestors
from notebook_helpers import extract_model_meta, prune_graph_safely
from esp_ppq_patch import apply_esp_ppq_patches
from esp_ppq_patch_2 import apply_addlut_patch
import esp_ppq_lut

seed_everything(SEED)
register_mod_op()
apply_esp_ppq_patches()
apply_addlut_patch()

# esp-ppq 1.3.x deduplica LUT y puede devolver el nombre canonico;
# el helper de la demo ignora ese retorno y deja un atributo ``lut`` que
# no existe en ExporterPatternInfo. Lo corregimos antes de exportar.
from esp_ppq.parser.espdl.export_patterns import AddLUTPattern, EspQuantType
from esp_ppq.parser.espdl.espdl_typedef import ExporterPatternInfo
from esp_ppq.IR.quantize import QuantableOperation

def _espdl_add_lut_export_fixed(self, op, graph, **kwargs):
    # Elimina referencias LUT de una pasada anterior antes de recalcular.
    op.attributes.pop('lut', None)
    quant_type = op.attributes.get('quant_type')
    if quant_type is None or quant_type == EspQuantType.F32 or not isinstance(op, QuantableOperation):
        return op
    info = ExporterPatternInfo()
    if not self.check_op(op):
        return op
    lut = None
    if quant_type == EspQuantType.S8:
        lut = self.calculate_lut(op, info, 127, -128, 1)
    elif quant_type == EspQuantType.S16:
        current_step = op.attributes.get('int16_lut_step', self.int16_step)
        if current_step is None or current_step <= 0:
            current_step = 256
        lut = self.calculate_lut(op, info, 2**15 - 1, -(2**15), int(current_step))
    if lut is not None:
        lut_name = self.get_lut_name(op, info)
        lut_name = info.add_lut(
            lut_name, lut, info.get_var_exponents(op.outputs[0].name)
        )
        op.attributes['lut'] = lut_name
    return op

AddLUTPattern.export = _espdl_add_lut_export_fixed
print('Aplicado fix de deduplicación de LUT para esp-ppq 1.3.x')
esp_ppq_lut.initialize(step=INT16_LUT_STEP, verbose=True)
print('Auxiliares oficiales listos; commit:', ESP_DL_REF)


In [ ]:
# @title 4. DataLoader de calibración (sin etiquetas)
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class CalibrationDataset(Dataset):
    """Dataset de calibración.

    ESP-DL y esp-ppq exigen una entrada NCHW de exactamente 3 canales.

    El tutorial oficial solo converts el modo "L", así que los PNG con canal alfa
    ("RGBA", 4 canales) y los de paleta ("P", 1 canal) llegaban al grafo tal cual
    y la primera Conv abortaba con:
        Given groups=1, weight of size [16, 3, 3, 3], expected input[1, 1, H, W]
        to have 3 channels, but got 1 channels instead
    Ver https://github.com/espressif/esp-dl/issues/329. Aquí la conversión a RGB es
    incondicional y el tensor resultante se valida antes de devolverlo, de modo
    que el error apunte al archivo culpable y no a la celda 7.
    """

    def __init__(self, paths, image_size):
        self.paths = list(paths)
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((image_size, image_size)),
            transforms.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0]),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        with Image.open(path) as handle:
            original_mode = handle.mode
            image = handle.convert('RGB')
        tensor = self.transform(image)
        if tensor.ndim != 3 or tensor.shape[0] != 3:
            raise RuntimeError(
                f'La imagen de calibración {path} no produjo un tensor RGB de 3 canales: '
                f'{tuple(tensor.shape)} (modo original: {original_mode}). '
                'Esp-PPQ solo admite entradas NCHW con 3 canales.'
            )
        return tensor

# Diagnostico previo: informa cuantas imagenes de calibracion no estan en RGB.
# Roboflow suele exportar PNG monocromos ("L"), con canal alfa ("RGBA") o de
# paleta ("P"); esos archivos son la causa del error de canales en la primera Conv.
# Ver https://github.com/espressif/esp-dl/issues/329.
non_rgb_images = []
for path in images:
    try:
        with Image.open(path) as handle:
            if handle.mode != 'RGB':
                non_rgb_images.append((path.name, handle.mode))
    except Exception as exc:
        raise RuntimeError(f'No se pudo abrir la imagen de calibracion {path}: {exc}') from exc
if non_rgb_images:
    print(f'[AVISO] {len(non_rgb_images)} imagen(es) de calibracion no estan en RGB; se convertiran a RGB:')
    for name, mode in non_rgb_images[:10]:
        print('   -', name, '(modo', mode, ')')
else:
    print('Todas las imagenes de calibracion estan en RGB.')

if len(images) > MAX_CALIB_IMAGES:
    images = random.Random(SEED).sample(images, MAX_CALIB_IMAGES)
calibration_dataset = CalibrationDataset(images, IMG_SIZE)
calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2 if DEVICE == 'cuda' else 0,
    pin_memory=(DEVICE == 'cuda'),
)
QATConfig.CALIB_STEPS = min(CALIB_STEPS, len(calibration_loader))
if QATConfig.CALIB_STEPS < 1:
    raise RuntimeError('El DataLoader de calibración no contiene batches.')

def ppq_collate(batch):
    # ESP-PPQ espera un tensor NCHW con exactamente 3 canales.
    if isinstance(batch, (list, tuple)):
        tensors = []
        for item in batch:
            if not torch.is_tensor(item):
                raise TypeError(f'Lote de calibración no contiene tensores: {type(item)}')
            tensors.append(item.unsqueeze(0) if item.ndim == 3 else item)
        batch = torch.cat(tensors, dim=0) if tensors else torch.empty(0)
    if not torch.is_tensor(batch):
        raise TypeError(f'Lote de calibración no compatible: {type(batch)}')
    if batch.ndim == 3:
        batch = batch.unsqueeze(0)
    if batch.ndim != 4:
        raise RuntimeError(f'Forma de calibración inesperada: {tuple(batch.shape)}')
    if batch.shape[1] != 3:
        if batch.shape[-1] == 3:
            batch = batch.permute(0, 3, 1, 2).contiguous()
        elif batch.shape[1] == 1:
            batch = batch.repeat(1, 3, 1, 1)
    if batch.shape[1] != 3:
        raise RuntimeError(
            f'La calibración debe entregar [batch, 3, H, W]; se recibió {tuple(batch.shape)}.'
        )
    return batch.to(device=DEVICE, dtype=torch.float32)

sample_batch = next(iter(calibration_loader))
print('Lote de calibración (antes de normalizar):', tuple(sample_batch.shape))
print('Lote de calibración (NCHW):', tuple(ppq_collate(sample_batch).shape))
print('Pasos efectivos de calibración:', QATConfig.CALIB_STEPS)


In [ ]:
# @title 5. Exportar best.pt a ONNX con el parche YOLO26
import onnx
import onnxruntime as ort

from export import ESP_YOLO, apply_export_patches
import inspect

# PyTorch 2.9+ puede seleccionar el exportador ONNX nuevo (Dynamo) por defecto.
# El exportador YOLO26/ESP-DL validado usa el exportador clásico; fijamos
# dynamo=False sin depender de la versión exacta de PyTorch de Colab.
try:
    _torch_onnx_has_dynamo = 'dynamo' in inspect.signature(torch.onnx.export).parameters
except (TypeError, ValueError):
    _torch_onnx_has_dynamo = False
if _torch_onnx_has_dynamo and not getattr(torch.onnx, '_espdl_export_patched', False):
    _original_torch_onnx_export = torch.onnx.export
    def _espdl_torch_onnx_export(*args, **kwargs):
        kwargs.setdefault('dynamo', False)
        return _original_torch_onnx_export(*args, **kwargs)
    torch.onnx.export = _espdl_torch_onnx_export
    torch.onnx._espdl_export_patched = True

def prepare_onnx_for_espdl():
    model = ESP_YOLO(QATConfig.PT_FILE)
    apply_export_patches(model)
    model.export(
        format='onnx',
        opset=13,
        simplify=True,
        imgsz=IMG_SIZE,
        dynamic=False,
        batch=1,
        device=0 if DEVICE == 'cuda' else 'cpu',
    )

prepare_onnx_for_espdl()
model_meta = extract_model_meta()
from ultralytics import YOLO
_yolo_for_metadata = YOLO(QATConfig.PT_FILE)
if isinstance(_yolo_for_metadata.names, dict):
    class_names = [_yolo_for_metadata.names[i] for i in sorted(_yolo_for_metadata.names.keys())]
else:
    class_names = list(_yolo_for_metadata.names)
del _yolo_for_metadata
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('metadata:', model_meta)
print('clases:', class_names)
if len(class_names) != int(model_meta['nc']):
    raise RuntimeError('El número de clases del best.pt no coincide con la metadata del detector.')

if int(model_meta.get('reg_max', -1)) != 1:
    raise ValueError(
        'El best.pt no parece ser YOLO26n RegMax=1. '
        'El tutorial oficial exige esta arquitectura; no se debe inventar un cambio de RegMax.'
    )

onnx_model = onnx.load(QATConfig.ONNX_PATH)
onnx.checker.check_model(onnx_model)
input_batch = onnx_model.graph.input[0].type.tensor_type.shape.dim[0].dim_value
if input_batch != 1:
    raise RuntimeError(f'ESP-DL requiere batch size 1; el ONNX exportó batch={input_batch}.')
input_dims = [d.dim_value for d in onnx_model.graph.input[0].type.tensor_type.shape.dim]
if len(input_dims) != 4 or input_dims[1] != 3:
    raise RuntimeError(
        f'El ONNX exportado no declara una entrada [1, 3, H, W]: {input_dims}. '
        'Regenere el ONNX antes de cuantizar; ESP-PPQ no puede corregir un eje de '
        'canales dinamico o erroneo.'
    )
print('Entradas ONNX:', [(x.name, [d.dim_value for d in x.type.tensor_type.shape.dim]) for x in onnx_model.graph.input])
print('Salidas ONNX:', [x.name for x in onnx_model.graph.output])
if len(onnx_model.graph.output) < 6:
    raise RuntimeError('La exportación no produjo las seis salidas esperadas de YOLO26.')

providers = ['CPUExecutionProvider']
if torch.cuda.is_available():
    providers.insert(0, 'CUDAExecutionProvider')
try:
    session = ort.InferenceSession(QATConfig.ONNX_PATH, providers=providers)
    print('ONNX Runtime providers:', session.get_providers())
except Exception as exc:
    print('Aviso: la validación ONNX Runtime no se pudo iniciar:', exc)


In [ ]:
# @title 6A. AutoQuant alternativo (opcional)
# Esta celda se ejecuta solo si RUN_MODE == 'autoquant'.
AUTOQUANT_RECORDS = []
AUTOQUANT_ONNX_PATH = None
AUTOQUANT_RESULT_DIR = None

if RUN_MODE.lower() == 'autoquant':
    from esp_ppq.api.setting import QuantizationSettingFactory

    def make_esp_ready_onnx(source, destination, nc):
        model = onnx.load(str(source))
        graph = model.graph

        def find_value_info(name):
            for collection in (graph.input, graph.value_info, graph.output):
                for item in collection:
                    if item.name == name:
                        return item
            return None

        def dims(name):
            info = find_value_info(name)
            if info is None or not info.type.HasField('tensor_type'):
                return []
            return [d.dim_value for d in info.type.tensor_type.shape.dim]

        def elem_type(name):
            info = find_value_info(name)
            if info is None or not info.type.HasField('tensor_type'):
                return onnx.TensorProto.FLOAT
            return info.type.tensor_type.elem_type

        # La rama one2many solo se usaba para validar; se elimina del grafo de despliegue.
        for output in list(graph.output):
            if any(output.name.startswith(prefix) for prefix in ('one2many_p3', 'one2many_p4', 'one2many_p5')):
                graph.output.remove(output)

        for target, box_name, cls_name in [
            ('one2one_p3', 'one2one_p3_box', 'one2one_p3_cls'),
            ('one2one_p4', 'one2one_p4_box', 'one2one_p4_cls'),
            ('one2one_p5', 'one2one_p5_box', 'one2one_p5_cls'),
        ]:
            output = next(
                (item for item in graph.output if item.name == target or item.name.endswith(target)),
                None,
            )
            if output is None:
                raise RuntimeError(f'No se encontró la salida ONNX {target}.')
            output_name = output.name
            producer = next((node for node in graph.node if output_name in node.output), None)
            if producer is None or producer.op_type != 'Concat':
                raise RuntimeError(f'{target} no es el Concat esperado del tutorial YOLO26.')
            box_input = None
            cls_input = None
            for input_name in producer.input:
                shape = dims(input_name)
                if 4 in shape:
                    box_input = input_name
                elif nc in shape:
                    cls_input = input_name
            if box_input is None or cls_input is None:
                if len(producer.input) < 2:
                    raise RuntimeError(f'No se pudo separar box/cls de {target}.')
                box_input = producer.input[0]
                cls_input = producer.input[1]
            graph.output.remove(output)
            for input_name, new_name in ((box_input, box_name), (cls_input, cls_name)):
                identity = onnx.helper.make_node(
                    'Identity', [input_name], [new_name], name=f'{target}_{new_name}_identity'
                )
                graph.node.append(identity)
                graph.output.append(
                    onnx.helper.make_tensor_value_info(new_name, elem_type(input_name), dims(input_name))
                )
            graph.node.remove(producer)

        onnx.checker.check_model(model)
        Path(destination).parent.mkdir(parents=True, exist_ok=True)
        onnx.save(model, str(destination))
        return destination

    AUTOQUANT_ONNX_PATH = ROOT / 'yolo26n_esp_ready.onnx'
    make_esp_ready_onnx(QATConfig.ONNX_PATH, AUTOQUANT_ONNX_PATH, int(model_meta['nc']))

    setting = AutoQuantSearchSetting(
        search_mode='fast',
        num_of_candidates=AUTOQUANT_CANDIDATES,
        score_direction='minimize',
        run_dir=str(ROOT / 'autoquant_results'),
        resume=False,
    )
    setting.top_strategy = 3
    setting.early_stop_patience = 2
    setting.strategy_space['weight_equalization'] = [False, True]
    setting.strategy_space['bias_correction'] = [False, True]
    setting.strategy_space['mixed_precision'] = [False, True] if USE_MIXED_INT16 else [False]
    setting.strategy_space['tqt'] = [False, True] if (USE_TQT_LUT and USE_MIXED_INT16) else [False]
    setting.strategy_space['blockwise_reconstruction'] = [False]
    setting.param_space['calib_algorithm']['method'] = ['percentile', 'mse']
    setting.param_space['tqt']['steps'] = [TQT_STEPS]
    setting.param_space['tqt']['lr'] = [2e-5]
    setting.param_space['tqt']['block_size'] = [1]
    setting.param_space['bias_correction']['steps'] = [32]
    setting.param_space['fusion_alignment']['elementwise_to'] = ['Align to Output']
    setting.param_space['fusion_alignment']['concat_to'] = ['Align to Output']
    setting.param_space['fusion_alignment']['avgpooling_to'] = ['Align to Output']
    setting.param_space['fusion_alignment']['resize_to'] = ['Align to Output']

    auto_device = DEVICE if (AUTOQUANT_USE_GPU and DEVICE == 'cuda') else 'cpu'
    AUTOQUANT_RECORDS = espdl_auto_quantize_onnx(
        onnx_import_file=str(AUTOQUANT_ONNX_PATH),
        espdl_export_file=str(OUT_DIR / 'yolo26n_autoquant_s3.espdl'),
        calib_dataloader=calibration_loader,
        calib_steps=QATConfig.CALIB_STEPS,
        input_shape=[3, IMG_SIZE, IMG_SIZE],
        target=TARGET,
        setting=setting,
        collate_fn=ppq_collate,
        device=auto_device,
        verbose=1,
    )
    if not AUTOQUANT_RECORDS:
        raise RuntimeError('AutoQuant no produjo candidatos; revise la compatibilidad del ONNX y los datos.')
    AUTOQUANT_RESULT_DIR = Path(AUTOQUANT_RECORDS[0]['folder'])
    model_files = sorted(AUTOQUANT_RESULT_DIR.glob('*.espdl'))
    if not model_files:
        raise RuntimeError('AutoQuant terminó sin un .espdl candidato.')
    FINAL_MODEL_PATH = OUT_DIR / f'yolo26n_{IMG_SIZE}_s8_s3_autoquant.espdl'
    shutil.copy2(model_files[0], FINAL_MODEL_PATH)
    for artifact in AUTOQUANT_RESULT_DIR.iterdir():
        if artifact.is_file() and artifact.suffix.lower() in {'.info', '.json', '.native'}:
            shutil.copy2(artifact, OUT_DIR / f'{FINAL_MODEL_PATH.stem}{artifact.suffix}')
    for artifact_name in ('candidates.json', 'summary.json'):
        artifact = ROOT / 'autoquant_results' / artifact_name
        if artifact.exists():
            shutil.copy2(artifact, OUT_DIR / f'autoquant_{artifact_name}')
    print('AutoQuant: candidato seleccionado:', FINAL_MODEL_PATH)
else:
    print('RUN_MODE != autoquant; se usará el pipeline YOLO26 completo de la celda 6B.')


In [ ]:
# @title 6B. Pipeline oficial YOLO26: PTQ + TQT + LUT
# Esta celda se ejecuta solo si RUN_MODE == 'full'.
if RUN_MODE.lower() == 'full':
    import esp_ppq.lib as PFL
    from esp_ppq.api.interface import load_onnx_graph
    from esp_ppq.core import QuantizationVisibility, TargetPlatform
    from esp_ppq.executor import TorchExecutor
    from esp_ppq.quantization.optim import (
        QuantizeSimplifyPass,
        QuantizeFusionPass,
        ParameterQuantizePass,
        RuntimeCalibrationPass,
        PassiveParameterQuantizePass,
        QuantAlignmentPass,
        TrainedQuantizationThresholdPass,
    )
    from esp_ppq_lut.passes import EspdlLUTFusionPass

    print('Cargando ONNX en ESP-PPQ...')
    graph = load_onnx_graph(onnx_import_file=QATConfig.ONNX_PATH)
    print('Entradas del grafo ESP-PPQ:', {name: var.shape for name, var in graph.inputs.items()})

    # esp-ppq 1.3.x NO lee la forma declarada en el ONNX: OnnxParser.build() crea
    # cada Variable con Variable(name=...) y sin shape, y tampoco lee value_info. La
    # unica fuente de la forma de la entrada es tracing_operation_meta, y
    # ParameterQuantizePass construye su tensor de entrada con executor.dummy_forward()
    # a partir de ese shape. Si la forma llegara a [1, 1, H, W], la primera Conv
    # abortaria con "to have 3 channels, but got 1 channels instead". Se fija aqui
    # para blindar esa ruta. Ver https://github.com/espressif/esp-dl/issues/325.
    if len(graph.inputs) != 1:
        raise RuntimeError(
            f'Se esperaba una unica entrada de grafo, hay {list(graph.inputs)}. '
            'Revise el ONNX exportado.'
        )
    graph_input_name, graph_input_var = next(iter(graph.inputs.items()))
    graph_input_var.shape = torch.Size([1, 3, IMG_SIZE, IMG_SIZE])
    print('Entrada del grafo fijada en', graph_input_name, tuple(graph_input_var.shape))
    output_names = list(graph.outputs.keys())
    if len(output_names) < 6:
        raise RuntimeError(f'Se esperaban seis salidas, se encontraron: {output_names}')

    aux_outputs = output_names[:3]
    main_outputs = output_names[3:6]
    aux_ops = get_exclusive_ancestors(graph, aux_outputs, main_outputs)
    main_ops = get_exclusive_ancestors(graph, main_outputs, aux_outputs)

    quantizer = PFL.Quantizer(platform=QATConfig.TARGET_PLATFORM, graph=graph)
    dispatching_table = PFL.Dispatcher(graph=graph, method='conservative').dispatch(
        quantizer.quant_operation_types
    )
    for opname, platform in list(dispatching_table.items()):
        if platform == TargetPlatform.UNSPECIFIED:
            dispatching_table[opname] = TargetPlatform(quantizer.target_platform)

    # La rama one2many no se despliega; se deja en FP32 solo para conservar el grafo
    # durante la calibración y luego se elimina.
    for op in aux_ops:
        if op.name in dispatching_table:
            dispatching_table[op.name] = TargetPlatform.FP32

    sensitive_names = set()
    if USE_MIXED_INT16:
        int16_platform = get_target_platform(TARGET, 16)
        for op in graph.operations.values():
            name = op.name
            if (
                '/one2one_cv2.' in name
                or '/one2one_cv3.' in name
                or re.search(r'/model\.(16|19|22)/cv2', name)
            ):
                sensitive_names.add(name)
                if name in dispatching_table:
                    dispatching_table[name] = int16_platform
    print('Operaciones promovidas a INT16:', len(sensitive_names))

    # Concat de la cabeza one2one: el tutorial lo deja en FP32 por la alineación.
    for op in main_ops:
        if op.type == 'Concat' and op.name in dispatching_table:
            dispatching_table[op.name] = TargetPlatform.FP32

    for op in graph.operations.values():
        platform = dispatching_table.get(op.name, TargetPlatform(quantizer.target_platform))
        quantizer.quantize_operation(op_name=op.name, platform=platform)

    executor = TorchExecutor(graph=graph, device=DEVICE)
    dummy_input = torch.zeros([1, 3, IMG_SIZE, IMG_SIZE], device=DEVICE)
    executor.tracing_operation_meta(inputs=dummy_input)
    traced_input = tuple(next(iter(graph.inputs.values())).shape or ())
    if traced_input != (1, 3, IMG_SIZE, IMG_SIZE):
        raise RuntimeError(
            f'El tracing no confirmo una entrada NCHW de 3 canales: {traced_input}. '
            'Aborte antes de lanzar PTQ/TQT para no perder tiempo de GPU.'
        )
    print('Tracing confirmado:', traced_input)

    passes = [
        QuantizeSimplifyPass(),
        QuantizeFusionPass(activation_type=quantizer.activation_fusion_types),
        ParameterQuantizePass(),
        RuntimeCalibrationPass(method=QATConfig.QUANT_CALIB_METHOD),
    ]
    if USE_TQT_LUT and USE_MIXED_INT16:
        passes.append(
            TrainedQuantizationThresholdPass(
                steps=QATConfig.TQT_STEPS,
                lr=QATConfig.TQT_LR,
                int_lambda=QATConfig.TQT_INT_LAMBDA,
                block_size=QATConfig.TQT_BLOCK_SIZE,
                collecting_device=QATConfig.TQT_COLLECTING_DEVICE,
            )
        )
    passes.extend([
        PassiveParameterQuantizePass(clip_visiblity=QuantizationVisibility.EXPORT_WHEN_ACTIVE),
        QuantAlignmentPass(elementwise_alignment=QATConfig.QUANT_ALIGNMENT),
    ])
    if USE_TQT_LUT and USE_MIXED_INT16:
        passes.append(EspdlLUTFusionPass(target_ops=['Swish'], lut_step=INT16_LUT_STEP))

    pipeline = PFL.Pipeline(passes)
    pipeline.optimize(
        calib_steps=QATConfig.CALIB_STEPS,
        collate_fn=ppq_collate,
        graph=graph,
        dataloader=calibration_loader,
        executor=executor,
    )
    if HAS_GPU:
        peak_gb = torch.cuda.max_memory_reserved() / (1024 ** 3)
        print(f'Pipeline PTQ/TQT/LUT terminado. Pico de VRAM reservada: {peak_gb:.2f} GiB')
        if TQT_COLLECTING_DEVICE == 'cuda' and peak_gb > 11.0:
            print(
                'AVISO: el pico se acerca al limite tipico de una T4 (16 GiB). '
                'Si la proxima ejecucion falla con OutOfMemoryError durante TQT, baje '
                'MAX_CALIB_IMAGES o ponga TQT_COLLECTING_DEVICE = "cpu".'
            )
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
else:
    print('RUN_MODE != full; se usará el resultado de AutoQuant.')


In [ ]:
# @title 7. Cirugía del grafo y exportación del .espdl
if RUN_MODE.lower() == 'full':
    FINAL_MODEL_PATH = OUT_DIR / f'yolo26n_{IMG_SIZE}_s8_s3.espdl'

if RUN_MODE.lower() == 'full':
    def resolve_output_name(graph, target):
        if target in graph.outputs:
            return target
        matches = [name for name in graph.outputs if target in name]
        if len(matches) == 1:
            return matches[0]
        raise RuntimeError(f'No se puede resolver la salida {target}: {list(graph.outputs)}')

    output_names = list(graph.outputs.keys())
    aux_names = [resolve_output_name(graph, name) for name in ('one2many_p3', 'one2many_p4', 'one2many_p5')]
    for name in aux_names:
        graph.outputs.pop(name, None)
    prune_graph_safely(graph)

    collected_outputs = {}
    for target, box_name, cls_name in [
        ('one2one_p3', 'one2one_p3_box', 'one2one_p3_cls'),
        ('one2one_p4', 'one2one_p4_box', 'one2one_p4_cls'),
        ('one2one_p5', 'one2one_p5_box', 'one2one_p5_cls'),
    ]:
        output_name = resolve_output_name(graph, target)
        original_var = graph.outputs[output_name]
        producer = original_var.source_op
        if producer is None or producer.type != 'Concat':
            raise RuntimeError(f'{output_name} no es un Concat separable.')
        box_var = None
        cls_var = None
        for input_var in producer.inputs:
            shape = list(input_var.shape or [])
            if 4 in shape:
                box_var = input_var
            elif int(model_meta['nc']) in shape:
                cls_var = input_var
        if box_var is None or cls_var is None:
            if len(producer.inputs) < 2:
                raise RuntimeError(f'No se pudo separar box/cls de {output_name}.')
            box_var, cls_var = producer.inputs[0], producer.inputs[1]

        graph.outputs.pop(output_name, None)
        for variable, new_name in ((box_var, box_name), (cls_var, cls_name)):
            old_name = variable.name
            graph.variables.pop(old_name, None)
            variable._name = new_name
            graph.variables[new_name] = variable
            collected_outputs[new_name] = variable
        graph.remove_operation(producer, keep_coherence=False)
        for variable in producer.inputs:
            if producer in variable.dest_ops:
                variable.dest_ops.remove(producer)

    final_order = [
        'one2one_p3_box', 'one2one_p3_cls',
        'one2one_p4_box', 'one2one_p4_cls',
        'one2one_p5_box', 'one2one_p5_cls',
    ]
    graph.outputs.clear()
    for name in final_order:
        graph.outputs[name] = collected_outputs[name]
    prune_graph_safely(graph)

    print('Salidas finales:', list(graph.outputs.keys()))
    exporter = PFL.Exporter(platform=QATConfig.TARGET_PLATFORM)
    export_kwargs = {'int16_lut_step': INT16_LUT_STEP} if (USE_TQT_LUT and USE_MIXED_INT16) else {}
    exporter.export(str(FINAL_MODEL_PATH), graph=graph, **export_kwargs)
else:
    print('Se conserva el .espdl seleccionado por AutoQuant:', FINAL_MODEL_PATH)


In [ ]:
#ES POSIBLE QUE ESTA CELDA NO SE EJECUTE CORRECTAMENTE, PERO NO PASA NADA, SIEMPRE Y CUANDO LA CELDA SIGUIENTE SI LO HAGA
# @title 8. Evaluación: inferencia emulada + mAP50-95 del grafo cuantizado
#
# Responde "¿cuánta precisión espero?" antes de flashear.
#
# NO se puede ejecutar un .espdl en Python: ESP-DL es C++ puro y el .espdl es un
# FlatBuffer que solo dl::Model sabe deserializar. Lo que sí se puede es ejecutar
# el GRAFO de ESP-PPQ, que sigue en memoria y emula el hardware bit a bit. Como
# esp_ppq_lut.initialize() registra lut_forward_provider en OPERATION_FORWARD_TABLE
# para todas las plataformas, los Swish convertidos en LUT por
# EspdlLUTFusionPass se simulan con la interpolación entera de dl_module_lut.hpp.
#
# Por qué NO se usa scripts/trainer.py + scripts/validator.py del tutorial:
#   1. No son mAP-free. validator.py decodifica con un Detect de Ultralytics y
#      end2end=True, en vez del top-K de YOLO26.
#   2. Asumen el formato [B, C, H, W] (NCHW) con box y cls pegados en un solo
#      Concat. Este contrato entrega 6 tensores NHWC separados: [1,64,64,4],
#      [1,64,64,1], [1,32,32,4], ... Ver esp_ppq_lut y yolo26.cpp ("// NHWC").
#      De ahí el ValueError del issue #336.
#   3. Además el cuaderno reordena graph.outputs en la celda 7, y
#      validator.py toma graph_outputs[3:6] por posición, lo que rompe.
#
# Aquí se decodifica con la misma lógica que el C++ (decode_grid) y el mAP se
# calcula con ap_per_class de Ultralytics, la misma función que usa model.val(),
# así que el número es comparable con el mAP FP32 de best.pt.
#
# Todo va envuelto en try/except: si algo falla, el modelo ya está exportado y
# descargado y el cuaderno no debe romperse.

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils.metrics import ap_per_class, box_iou
try:
    from esp_ppq import QuantizationPolicy
except ImportError:
    from esp_ppq.core.quant import QuantizationPolicy

EVAL_MAX_IMAGES = 200          # tope de imágenes de validación para acotar el tiempo
# El umbral debe ser el de Ultralytics en modo val (default conf=0.001), NO el
# yolo_threshold=0.1 del firmware. Con 0.25 las detecciones por debajo se
# contarian como acertadas y el mAP saldria inflado. El firmware con 0.1 verá
# algo menos de recall que este numero.
EVAL_CONF = 0.001
EVAL_TOP_K = 100               # máximo de detecciones por imagen (Ultralytics val usa 300)
EVAL_IOU_THRESHOLDS = torch.linspace(0.50, 0.95, 10).numpy()   # los 10 de Ultralytics
OUTPUT_NAMES = [
    'one2one_p3_box', 'one2one_p3_cls',
    'one2one_p4_box', 'one2one_p4_cls',
    'one2one_p5_box', 'one2one_p5_cls',
]
EVAL_RESULTS = {}

if RUN_MODE.lower() == 'full' and 'graph' in globals() and not DATA_YAML.exists():
    print('AVISO: no se encontro data.yaml; se omite la evaluacion. DATA_YAML =', DATA_YAML)


def espdl_letterbox(img_rgb, size):
    """Replica de ImagePreprocessor con letterbox: resize NEAREST, pad 114 centrado.

    El reshape usa min(size/w, size/h) y el pad se reparte igual, igual que
    managed_components/espressif__esp-dl/vision/image/dl_image_preprocessor.cpp.
    Devuelve el tensor NCHW float en [0, 1] (lo cuantiza ESP-PPQ con la config
    del tensor de entrada, que es la misma que usa el C++) y la transformacion
    inversa, para deshacer el letterbox en las cajas.
    """
    src_h, src_w = img_rgb.shape[:2]
    scale = min(size / src_w, size / src_h)
    new_w, new_h = int(src_w * scale), int(src_h * scale)
    t = torch.from_numpy(np.ascontiguousarray(img_rgb)).permute(2, 0, 1).unsqueeze(0).float()
    resized = F.interpolate(t, size=(new_h, new_w), mode='nearest')
    canvas = resized.new_full((1, 3, size, size), 114.0)
    pad_top = (size - new_h) // 2
    pad_left = (size - new_w) // 2
    canvas[:, :, pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized
    return canvas / 255.0, scale, pad_left, pad_top


def decode_one2one(outs, scale, pad_left, pad_top, src_w, src_h, conf_thresh, top_k):
    """decode_grid() en Python: 6 tensores NHWC -> cajas NMS-free en la imagen original.

    Clonica esp-dl/models/yolo26/yolo26.cpp: score = sigmoid(cls), la caja se
    decodifica como x1=(cx-d_l)*stride con cx = w + 0.5, y el top-K es por score
    sin NMS porque la cabeza one2one de YOLO26 ya es NMS-free.
    """
    strides = (8, 16, 32)
    levels = ('p3', 'p4', 'p5')
    chunks = []
    for stride, level in zip(strides, levels):
        box_nhwc = outs[f'one2one_{level}_box'].float()                  # [1, G, G, 4]
        grid = box_nhwc.shape[1]      # NHWC: el lado es shape[1], igual que en yolo26.cpp
        box = box_nhwc.reshape(-1, 4)                                    # (d_l, d_t, d_r, d_b)
        cls = outs[f'one2one_{level}_cls'].float().reshape(-1, outs[f'one2one_{level}_cls'].shape[-1])
        best_score, best_cls = torch.sigmoid(cls).max(dim=1)
        keep = best_score >= conf_thresh
        if not bool(keep.any()):
            continue
        idx = torch.nonzero(keep, as_tuple=False).squeeze(1)
        d = box[keep]
        # pixel_idx = h * grid + w, igual que el bucle anidado de yolo26.cpp
        cx = (idx % grid).float() + 0.5                               # w + 0.5
        cy = torch.div(idx, grid, rounding_mode='floor').float() + 0.5   # h + 0.5
        # Deshacer el letterbox: espacio del modelo -> imagen original
        x1 = ((cx - d[:, 0]) * stride - pad_left) / scale
        y1 = ((cy - d[:, 1]) * stride - pad_top) / scale
        x2 = ((cx + d[:, 2]) * stride - pad_left) / scale
        y2 = ((cy + d[:, 3]) * stride - pad_top) / scale
        x1 = x1.clamp(0.0, float(src_w))
        y1 = y1.clamp(0.0, float(src_h))
        x2 = x2.clamp(0.0, float(src_w))
        y2 = y2.clamp(0.0, float(src_h))
        valid = (x2 > x1) & (y2 > y1)
        if not bool(valid.any()):
            continue
        chunks.append(torch.stack([
            x1[valid], y1[valid], x2[valid], y2[valid],
            best_score[valid], best_cls[valid].float(),
        ], dim=1))
    if not chunks:
        return torch.zeros((0, 6))
    dets = torch.cat(chunks, dim=0)
    if dets.shape[0] > top_k:
        dets = dets[dets[:, 4].argsort(descending=True)[:top_k]]
    return dets


def resolve_eval_paths(val_entry, yaml_path):
    """Normaliza data['val'] de Ultralytics a una lista de rutas de imagen.

    check_det_dataset() NO siempre devuelve una lista. Con el data.yaml de
    Roboflow el campo 'val' puede venir como un string: un directorio, un
    .txt con la lista de archivos, o una ruta relativa al propio yaml. Hacer
    [str(p) for p in data['val']] sobre un string recorre sus CARACTERES, y
    por eso la celda 8 moria con FileNotFoundError: 'e' (el primer caracter de
    la ruta, tipicamente 'e' de 'exported/...' o 'eval/...').

    Acepta lista/tupla/set, str, Path, o una lista de un solo elemento.
    Reutiliza find_images()/IMAGE_EXTENSIONS de la celda 2.
    """
    yaml_path = Path(yaml_path)

    def normalize_one(item):
        candidate = Path(item)
        if not candidate.is_absolute():
            # Roboflow escribe rutas relativas al directorio del data.yaml.
            candidate = yaml_path.parent / candidate
        if candidate.is_dir():
            return [str(p) for p in find_images(candidate)]
        if candidate.is_file() and candidate.suffix.lower() == '.txt':
            return [
                line.strip()
                for line in candidate.read_text(encoding='utf-8').splitlines()
                if line.strip() and not line.strip().startswith('#')
            ]
        if candidate.is_file():
            return [str(candidate)]
        return [str(p) for p in find_images(candidate)]  # puede ser un glob

    if val_entry is None:
        raise RuntimeError("El data.yaml no define la particion 'val'.")
    if isinstance(val_entry, (str, Path)):
        raw = normalize_one(val_entry)
    elif isinstance(val_entry, (list, tuple, set)):
        raw = []
        for item in val_entry:
            raw.extend(normalize_one(item))
    else:
        raise RuntimeError(f"Tipo inesperado para data['val']: {type(val_entry)}")
    # Deduplicar conservando orden: una misma imagen puede entrar dos veces si
    # el yaml lista a la vez un archivo y la carpeta que lo contiene, y eso
    # inflaria el mAP al contar dos veces el mismo target.
    seen, unique = set(), []
    for path in raw:
        key = str(Path(path).resolve())
        if key not in seen:
            seen.add(key)
            unique.append(path)
    return unique


def find_label_file(image_path):
    """Localiza el .txt de etiquetas YOLO de una imagen, o None.

    Roboflow exporta images/ y labels/ como carpetas HERMANAS, asi que el
    .txt no esta junto al .jpg: para .../valid/images/x.jpg esta en
    .../valid/labels/x.txt. Con solo with_suffix('.txt') no se encontraba
    ningun archivo, total_gt se quedaba en 0 y la celda abortaba con el
    RuntimeError de 'el conjunto de validacion no tiene ninguna etiqueta'.
    """
    image_path = Path(image_path)
    parent = image_path.parent
    candidates = [image_path.with_suffix('.txt')]
    if parent.name.lower() in ('images', 'imgs', 'imageset'):
        candidates.append(parent.parent / 'labels' / (image_path.stem + '.txt'))
    else:
        candidates.append(parent / 'labels' / (image_path.stem + '.txt'))
        candidates.append(parent.parent / 'labels' / (image_path.stem + '.txt'))
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    return None


def read_yolo_label(image_path, src_w, src_h):
    """Etiqueta YOLO x_center y_center w h normalizada -> (M,5) [cls,x1,y1,x2,y2] en pixeles.

    Si no se encuentra el .txt, la imagen cuenta como negativa.
    """
    txt = find_label_file(image_path)
    if txt is None:
        return torch.zeros((0, 5))
    rows = []
    for line in txt.read_text(encoding='utf-8').splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = float(parts[0])
        xc, yc, w, h = (float(v) for v in parts[1:5])
        rows.append([
            cls,
            (xc - w / 2) * src_w, (yc - h / 2) * src_h,
            (xc + w / 2) * src_w, (yc + h / 2) * src_h,
        ])
    if not rows:
        return torch.zeros((0, 5))
    return torch.tensor(rows, dtype=torch.float32)


def match_detections(gt, dets, thresholds):
    """Matching codicioso por umbral de IoU (equivalente a BaseValidator.update_metrics).

    dets debe venir ordenado por score descendente, que es como lo produce
    decode_one2one. Devuelve (N, len(thresholds)) con 1.0 en los umbrales donde
    la deteccion empareja con un target libre.
    """
    n = dets.shape[0]
    tp = np.zeros((n, len(thresholds)), dtype=np.float32)
    if n == 0 or gt.shape[0] == 0:
        return tp
    iou = box_iou(gt[:, 1:5], dets[:, :4]).cpu().numpy()
    for j, thr in enumerate(thresholds):
        used = np.zeros(gt.shape[0], dtype=bool)
        for k in range(n):
            col = iou[:, k].copy()
            col[used] = -1.0
            b = int(col.argmax())
            if col[b] >= thr and gt[b, 0] == dets[k, 5]:
                tp[k, j] = 1.0
                used[b] = True
    return tp


def draw_detections(img_rgb, dets, title, out_path):
    """Anotacion minima, mismo estilo que eval_espdl_model del tutorial."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches

    src_h, src_w = img_rgb.shape[:2]
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(img_rgb)
    for row in dets:
        color = '#FF3333' if int(row[5]) == 0 else '#3399FF'
        ax.add_patch(patches.Rectangle(
            (row[0], row[1]), row[2] - row[0], row[3] - row[1],
            linewidth=2, edgecolor=color, facecolor='none'))
        ax.text(row[0], max(row[1] - 6, 0), f'{class_names[int(row[5])]} {row[4]:.2f}',
                color='white', fontsize=11, fontweight='bold',
                bbox=dict(facecolor=color, alpha=0.75, lw=0, pad=2))
    ax.set_xlim(0, src_w)
    ax.set_ylim(src_h, 0)
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold')
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    return out_path


def espppq_quant_config(variable):
    """Devuelve el TensorQuantizationConfig de una variable del grafo, o None.

    graph.outputs[name] devuelve una QuantableVariable. ESA clase NO tiene
    get_dtype_string(): ese metodo pertenece a dl::TensorBase del runtime C++
    (ver yolo_inspect_io_tensors() en main/yolo_bridge.cpp), no a la capa IR de
    ESP-PPQ. En ESP-PPQ la config de cuantizacion de una variable vive en el
    TQC de salida de su op productor, expuesto por la propiedad
    QuantableVariable.source_op_config. Puede ser None si la variable no tiene
    productor o si ese productor no es QuantableOperation (pasa tras la cirugia
    del grafo de la celda 7).
    """
    if getattr(variable, 'source_op', None) is None:
        return None
    getter = getattr(variable, 'source_op_config', None)
    if getter is None:
        return None
    try:
        # source_op_config es una @property, asi que getattr ya devuelve el
        # TQC. Se admite metodo por si otra version de ESP-PPQ lo expone asi.
        return getter() if callable(getter) else getter
    except (AttributeError, IndexError, ValueError, TypeError):
        return None


def espppq_dtype_string(variable):
    """Describe el tipo numerico de una variable del grafo ESP-PPQ.

    Reemplaza a QuantableVariable.get_dtype_string(), que no existe. Deduce el
    ancho de bits del rango cuantizado (quant_max - quant_min + 1) en vez de
    asumirlo, y cae al DataType de almacenamiento si no hay TQC.
    """
    cfg = espppq_quant_config(variable)
    if cfg is None:
        return f'{getattr(variable, "dtype", "?")} (sin TQC)'

    # OJO: TQC.policy devuelve el _policy crudo. En este fork de ESP-PPQ puede
    # ser un int (mascara de bits) o un QuantizationPolicy segun como se haya
    # construido el grafo, y solo el segundo tiene to_dict(). Se normaliza.
    raw_policy = cfg.policy
    try:
        policy = (raw_policy if hasattr(raw_policy, 'to_dict')
                  else QuantizationPolicy(int(raw_policy))).to_dict()
    except Exception:
        policy = {}

    if policy.get('FLOATING'):
        kind = f'FP{cfg.quant_max}'
    else:
        # (span - 1).bit_length() y no span.bit_length() - 1: el rango
        # simetrico de ESP-PPQ es [-127, 127] (span = 255) y con la otra
        #        formula daria 7 en vez de 8. Equivale a ceil(log2(span)).
        span = int(cfg.quant_max) - int(cfg.quant_min) + 1
        bits = max(1, (span - 1).bit_length()) if span > 1 else 1
        kind = f'INT{bits}'

    grain = 'per-channel' if policy.get('PER_CHANNEL') else 'per-tensor'
    try:
        state = cfg.state.name
    except Exception:
        state = '?'
    return f'{kind} {grain} [{int(cfg.quant_min)}..{int(cfg.quant_max)}] {state}'


if RUN_MODE.lower() == 'full' and 'graph' in globals() and DATA_YAML.exists():
    eval_executor = TorchExecutor(graph=graph, device=DEVICE)

    try:
        # Comprobacion de contrato antes de nada: formas y dtypes de las 6 salidas.
        for name in OUTPUT_NAMES:
            tensor = graph.outputs[name]
            print(f'  {name:18s} shape={tuple(tensor.shape)} '
                  f'dtype={espppq_dtype_string(tensor)}')
        print()

        data_info = check_det_dataset(str(DATA_YAML))
        # No usar [str(p) for p in data_info['val']] a ciegas: 'val' puede ser
        # un string y eso recorreria sus caracteres. Ver resolve_eval_paths().
        raw_val = data_info.get('val')
        print(f"data['val'] crudo: {type(raw_val).__name__} = {str(raw_val)[:160]}")
        val_paths = resolve_eval_paths(raw_val, DATA_YAML)
        random.Random(SEED).shuffle(val_paths)
        val_paths = val_paths[:EVAL_MAX_IMAGES]
        eval_names = {i: str(n) for i, n in enumerate(class_names)}
        print(f'\nValidacion: {len(val_paths)} imagenes | clases: {eval_names}')
        if not val_paths:
            raise RuntimeError(
                f"No se encontro ninguna imagen de validacion. data['val'] = "
                f'{raw_val!r} y el data.yaml esta en {DATA_YAML}.')
        labelled = sum(1 for p in val_paths if find_label_file(p) is not None)
        print(f'De esas, {labelled} tienen .txt de etiquetas; '
              f'{len(val_paths) - labelled} cuentan como negativas.')

        # ------------------------------------------------ 1. Inferencia visual
        preview_dir = ROOT / 'eval_results'
        preview_dir.mkdir(parents=True, exist_ok=True)
        print('\n=== INFERENCIA VISUAL EMULADA (grafo cuantizado) ===')
        for path in val_paths[:3]:
            img_rgb = np.array(Image.open(path).convert('RGB'))
            src_h, src_w = img_rgb.shape[:2]
            inp, sc, pl, pt = espdl_letterbox(img_rgb, IMG_SIZE)
            outs = eval_executor.forward(
                inputs=inp.to(DEVICE), output_names=OUTPUT_NAMES)
            dets = decode_one2one(
                dict(zip(OUTPUT_NAMES, outs)), sc, pl, pt, src_w, src_h,
                EVAL_CONF, EVAL_TOP_K)
            out_png = preview_dir / f'{Path(path).stem}_{IMG_SIZE}_s8_s3.png'
            draw_detections(img_rgb, dets,
                             f'YOLO26n {IMG_SIZE} INT8/LUT16 | {Path(path).name}', out_png)
            print(f'  {Path(path).name}: {dets.shape[0]} detecciones '
                  f'({src_w}x{src_h}, scale={sc:.3f}) -> {out_png.name}')
            for row in dets[:5]:
                print(f'     {class_names[int(row[5])]} score={row[4]:.2f} '
                      f'[{row[0]:.0f},{row[1]:.0f},{row[2]:.0f},{row[3]:.0f}]')
        print('Imagenes anotadas en:', preview_dir)
        try:
            from IPython.display import display as ipy_display, Image as ipy_image
            for path in val_paths[:3]:
                ipy_display(ipy_image(filename=str(preview_dir / f'{Path(path).stem}_{IMG_SIZE}_s8_s3.png')))
        except Exception:
            pass

        # ------------------------------------------------ 2. mAP50-95 emulado
        print('\n=== mAP50-95 EMULADO (grafo cuantizado) ===')
        tp_list, conf_list, pred_cls_list, target_cls_list = [], [], [], []
        total_gt = 0
        bar = tqdm(val_paths, desc='Evaluacion cuantizada', unit='img')
        for path in bar:
            try:
                img_rgb = np.array(Image.open(path).convert('RGB'))
            except Exception:
                continue
            src_h, src_w = img_rgb.shape[:2]
            inp, sc, pl, pt = espdl_letterbox(img_rgb, IMG_SIZE)
            outs = eval_executor.forward(
                inputs=inp.to(DEVICE), output_names=OUTPUT_NAMES)
            dets = decode_one2one(
                dict(zip(OUTPUT_NAMES, outs)), sc, pl, pt, src_w, src_h,
                EVAL_CONF, EVAL_TOP_K)
            gt = read_yolo_label(path, src_w, src_h)
            total_gt += int(gt.shape[0])
            target_cls_list.extend(gt[:, 0].cpu().numpy().tolist())
            if dets.shape[0] == 0:
                continue
            tp = match_detections(gt, dets, EVAL_IOU_THRESHOLDS)
            if tp.shape[0] == 0:
                continue
            tp_list.append(tp)
            conf_list.append(dets[:, 4].cpu().numpy())
            pred_cls_list.append(dets[:, 5].cpu().numpy())
            bar.set_postfix(gt=total_gt, det=int(sum(t.shape[0] for t in tp_list)))

        if not target_cls_list:
            raise RuntimeError(
                'El conjunto de validacion no tiene ninguna etiqueta. Revisa que '
                'data.yaml apunte a la particion val y que las imagenes tengan .txt.')

        tp_all = np.concatenate(tp_list, axis=0) if tp_list else np.zeros((0, 10), dtype=np.float32)
        conf_all = np.concatenate(conf_list) if conf_list else np.zeros(0)
        pred_cls_all = np.concatenate(pred_cls_list) if pred_cls_list else np.zeros(0)
        target_cls_all = np.array(target_cls_list)

        _, _, p_at_f1, r_at_f1, f1_at_f1, ap, ap_classes, *_ = ap_per_class(
            tp_all, conf_all, pred_cls_all, target_cls_all, plot=False, names=eval_names)

        map50 = float(ap[:, 0].mean()) if ap.size else 0.0
        map5095 = float(ap.mean()) if ap.size else 0.0
        print(f'\nImagenes evaluadas : {len(val_paths)}')
        print(f'Targets totales    : {total_gt}')
        print(f'Detecciones        : {int(tp_all.shape[0])}')
        print(f'mAP@0.5            : {map50:.4f}')
        print(f'mAP@0.5:0.95       : {map5095:.4f}')
        if ap.size:
            print(f'P (max-F1)         : {float(p_at_f1.mean()):.4f}')
            print(f'R (max-F1)         : {float(r_at_f1.mean()):.4f}')
        EVAL_RESULTS = {
            'emulated_map50': round(map50, 5),
            'emulated_map50_95': round(map5095, 5),
            'precision_at_max_f1': round(float(p_at_f1.mean()), 5) if ap.size else None,
            'recall_at_max_f1': round(float(r_at_f1.mean()), 5) if ap.size else None,
            'images_evaluated': len(val_paths),
            'targets_total': total_gt,
            'detections_total': int(tp_all.shape[0]),
            'conf_threshold': EVAL_CONF,
            'note': 'Grafo ESP-PPQ con emulacion de LUT INT16. No es el .espdl ejecutado '
                    'en el chip, pero reproduce el pipeline de cuantizacion completo.',
        }
        print('\nComo leerlo: este es el mAP del GRAFO cuantizado, comparable con el '
              'YOLO(best.pt).val() en FP32. El firmware usa yolo_threshold=0.1 y K=32, '
              'asi que en el chip veras algo menos de recall. La cabeza one2one es '
              'NMS-free, por eso aqui no se aplica NMS y el C++ tampoco.')
    except Exception as exc:
        import traceback
        traceback.print_exc()
        print(f'\nAVISO: la evaluacion fallo ({type(exc).__name__}: {exc}). '
              'El .espdl ya fue exportado y descargado; esto es solo informative.')
else:
    print('RUN_MODE != full o sin data.yaml; se omite la evaluacion.')


In [ ]:
# @title 9. Comprobación, reporte y descarga
import hashlib
import importlib.metadata

package_versions = {}
for package_name in ('torch', 'torchvision', 'ultralytics', 'esp-ppq', 'onnx', 'onnxscript', 'onnxruntime-gpu', 'onnxruntime'):
    try:
        package_versions[package_name] = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        pass

if not FINAL_MODEL_PATH.exists():
    raise FileNotFoundError(f'No se encontró el modelo final: {FINAL_MODEL_PATH}')

if RUN_MODE.lower() == 'full' and 'graph' in globals():
    # Se reutiliza ppq_collate en vez de indexar el lote a mano: default_collate
    # puede devolver una lista o un tensor apilado segun la version de torchvision,
    # y ppq_collate garantiza NCHW de 3 canales con el device y dtype correctos.
    sample = ppq_collate(next(iter(calibration_loader)))[:1]
    if tuple(sample.shape) != (1, 3, IMG_SIZE, IMG_SIZE):
        raise RuntimeError(f'Lote de previsualizacion con forma inesperada: {tuple(sample.shape)}')
    print('Muestra de previsualizacion:', tuple(sample.shape))
    quantized_outputs = TorchExecutor(graph=graph, device=DEVICE).forward(sample)
    print('Salidas cuantificadas (previsualización):')
    for name, value in zip(list(graph.outputs.keys()), quantized_outputs):
        print(' ', name, tuple(value.shape))

model_sha256 = hashlib.sha256(FINAL_MODEL_PATH.read_bytes()).hexdigest()
report = {
    'weights': str(WEIGHTS_PATH),
    'mode': RUN_MODE,
    'target': TARGET,
    'input_size': IMG_SIZE,
    'device_used': DEVICE,
    'package_versions': package_versions,
    'autoquant_candidates': AUTOQUANT_RECORDS,
    'calibration_images': len(calibration_dataset),
    'calibration_batches_used': QATConfig.CALIB_STEPS,
    'calibration_steps': QATConfig.CALIB_STEPS,
    'mixed_int16': USE_MIXED_INT16,
    'tqt_lut': USE_TQT_LUT and USE_MIXED_INT16,
    'model_meta': model_meta,
    'class_names': class_names,
    'output_names': (
        list(graph.outputs.keys()) if RUN_MODE.lower() == 'full' and 'graph' in globals()
        else [x.name for x in onnx.load(str(AUTOQUANT_ONNX_PATH or QATConfig.ONNX_PATH)).graph.output]
    ),
    'onnx_calibration_layout': 'NCHW',
    'esp_dl_runtime_layout': 'NHWC para el procesador YOLO26 oficial',
    'model_file': str(FINAL_MODEL_PATH),
    'model_size_bytes': FINAL_MODEL_PATH.stat().st_size,
    'current_app_partition_bytes': CURRENT_APP_PARTITION_BYTES,
    'fits_current_app_partition': FINAL_MODEL_PATH.stat().st_size <= CURRENT_APP_PARTITION_BYTES,
    'model_sha256': model_sha256,
    'esp_dl_runtime_requirement': (
        'ESP-DL >= 3.3.11 (3.3.12 recomendado) para el formato/tutorial YOLO26 actual'
        if USE_MIXED_INT16 else
        'ESP-DL >= 3.3.10; validar el formato .espdl en el runtime usado'
    ),
    'emulated_evaluation': EVAL_RESULTS or 'no ejecutada',
    'firmware_compatibility_warning': (
        'El firmware actual del proyecto espera una salida output0 y preprocesado manual; '
        'este modelo oficial expone seis salidas one2one_* y requiere el procesador YOLO26 de ESP-DL.'
    ),
}
report_path = OUT_DIR / 'quantization_report.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
print('Reporte:', report_path)
print('Modelo:', FINAL_MODEL_PATH, 'bytes=', FINAL_MODEL_PATH.stat().st_size)
if FINAL_MODEL_PATH.stat().st_size > CURRENT_APP_PARTITION_BYTES:
    print('ADVERTENCIA: el modelo no cabe en la partición app0 actual de 5 MiB; requiere otra partición/estrategia de flash.')
print('SHA256:', model_sha256)

package_base = ROOT / f'yolo26n_{IMG_SIZE}_s8_s3_package'
package_zip = shutil.make_archive(str(package_base), 'zip', root_dir=OUT_DIR)
print('ZIP:', package_zip)
if DOWNLOAD_RESULT:
    try:
        from google.colab import files
        files.download(package_zip)
    except Exception as exc:
        print('No se pudo iniciar la descarga automática; usa el panel de archivos:', exc)
